<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/Correction_language_models_and_the_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installez les bibliothèques nécessaires à ce notebook (Keras et keras-hub) en les mettant à jour vers les versions les plus récentes.

In [ ]:
!pip install keras keras-hub --upgrade -q

**Consigne :**  
Configurez l’environnement pour que Keras utilise le backend **JAX** (via une variable d’environnement), avant d’importer/initialiser Keras.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

`os.environ["KERAS_BACKEND"] = "jax"`

Force Keras à utiliser JAX comme moteur de calcul.  
Il faut le faire avant d’importer ou d’initialiser Keras, sinon le backend déjà chargé ne changera pas correctement.

Créez une *cell magic* Jupyter (par exemple `%%backend <nom_backend>`) qui exécute le contenu de la cellule uniquement si la variable d’environnement `KERAS_BACKEND` correspond au backend demandé ; sinon, elle affiche un message expliquant comment changer le backend et relancer le runtime.

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

`@register_cell_magic`
`def backend(line, cell): ...`

Crée une magic Jupyter du type `%%backend jax`.  
L’idée est simple : si le backend demandé dans la cellule correspond bien à `KERAS_BACKEND`, le contenu de la cellule s’exécute.  
Sinon, la cellule ne tourne pas et affiche un message pour rappeler qu’il faut changer la variable d’environnement puis redémarrer le runtime.

## Modèle de langage “Shakespeare” (RNN)

Téléchargez le dataset texte “Shakespeare” depuis une URL fournie et chargez son contenu complet dans une variable Python (chaîne de caractères).

In [ ]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespeare = open(filename, "r").read()

Affichez les ~250 premiers caractères du corpus afin de vérifier le chargement et observer le format du texte.

In [ ]:
shakespeare[:250]

C’est aussi utile pour voir à quoi ressemble le format du texte avant de le découper.

Découpez le corpus en séquences de longueur fixe (par exemple 100 caractères).  
Créez :
- des **features** : segments du texte,
- des **labels** : les mêmes segments mais décalés d’un caractère (prédire le caractère suivant).  
Construisez ensuite un `tf.data.Dataset` de paires (features, labels).

In [ ]:
import tensorflow as tf

sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

`def split_input(...)`
`features = ...`
`labels = ...`
`dataset = tf.data.Dataset.from_tensor_slices((features, labels))`

Découpe le texte en morceaux de longueur fixe ici `100`.  
`features` contient un segment du texte, et `labels` contient le même segment décalé d’un caractère.  
Donc pour chaque position, le modèle apprend à prédire le caractère suivant.  
Le tout est ensuite converti en `tf.data.Dataset` pour préparer l’entraînement.

Récupérez un premier batch (ou un premier exemple) du dataset et comparez les 50 premiers caractères de l’entrée et de la sortie pour vérifier que la sortie correspond bien à l’entrée décalée d’un caractère.

In [ ]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

`x, y = next(dataset.as_numpy_iterator())`
`x[:50], y[:50]`

Prend un premier exemple du dataset pour vérifier visuellement que la sortie est bien l’entrée décalée d’un caractère.  
Si tout est correct, chaque caractère de `y` doit correspondre au caractère suivant dans `x`.

Créez un tokenizer Keras basé sur `TextVectorization` qui :
- ne standardise pas le texte (`standardize=None`),
- découpe en **caractères**,
- renvoie des séquences de longueur `sequence_length`.  
Adaptez-le sur le texte du dataset.

In [ ]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

`tokenizer = layers.TextVectorization(...)`
`tokenizer.adapt(...)`

Crée un tokenizer caractère par caractère.  
`standardize=None` évite de modifier le texte, donc la casse et la ponctuation sont conservées.  
`split="character"` découpe en caractères et non en mots.  
`adapt(...)` sert à apprendre le vocabulaire présent dans le corpus.

Affichez la taille du vocabulaire appris par votre tokenizer (nombre de caractères distincts).

In [ ]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

Transformez le dataset texte en dataset d’entiers en appliquant le tokenizer sur les features et les labels.  
Préparez ensuite un pipeline d’entraînement : mélange (shuffle), mini-batchs, et mise en cache.

In [ ]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

`dataset = dataset.map(...)`
`training_data = dataset.shuffle(...).batch(...).cache()`

Convertit les chaînes de caractères en séquences d’IDs entiers grâce au tokenizer.  
Puis prépare un pipeline d’entraînement classique :
- `shuffle` mélange les exemples,
- `batch` regroupe en mini-lots,
- `cache` évite de recalculer inutilement les transformations.

Construisez un modèle de langage caractère par caractère avec Keras :
- entrée : séquence d’IDs de tokens,
- `Embedding`,
- une couche `GRU` avec `return_sequences=True`,
- `Dropout`,
- couche `Dense` de sortie de taille `vocabulary_size` avec `softmax`.  
Utilisez l’API fonctionnelle et stockez le modèle dans une variable.

In [ ]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

`inputs = layers.Input(...)`
`x = layers.Embedding(...)`
`x = layers.GRU(...)`
`x = layers.Dropout(...)`
`outputs = layers.Dense(..., activation="softmax")`
`model = keras.Model(inputs, outputs)`

Construit le modèle de langage caractère par caractère.  
L’`Embedding` transforme chaque ID en vecteur dense.  
Le `GRU` lit la séquence dans l’ordre et garde une mémoire du contexte précédent.  
Le `Dropout` régularise un peu le modèle.  
La couche finale `Dense + softmax` donne, pour chaque position, une probabilité sur tous les caractères du vocabulaire.

Dans notre modèle de génération, nous utilisons une couche :

```python
layers.GRU(hidden_dim, return_state=True)
````

Un **GRU (Gated Recurrent Unit)** est un type de réseau de neurones récurrent (RNN).
Comme tous les RNN, il traite une séquence **pas à pas** (token après token) et conserve une **mémoire interne** appelée *état caché*.

À chaque étape :

* il reçoit un token,
* il met à jour son **état interne**,
* il produit une sortie.

Un GRU possède **un seul état caché**, noté généralement ( h ), de dimension `hidden_dim`.

`return_state=True` demande à récupérer cet état caché en plus de la sortie normale.  
C’est utile pour la génération auto-régressive, où on veut réinjecter l’état d’un pas au suivant.

Affichez l’architecture du modèle (noms des couches, dimensions, nombre de paramètres) avec `model.summary`.

In [ ]:
model.summary(line_length=80)

C’est surtout une vérification de cohérence avant l’entraînement.

## Génération de texte (Shakespeare)

Compilez le modèle avec :
- optimiseur Adam,
- perte `sparse_categorical_crossentropy`,
- métrique `sparse_categorical_accuracy`,  
puis entraînez-le pendant un certain nombre d’époques (ex : 20) sur le dataset préparé.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=20)

`model.compile(...)`
`model.fit(training_data, epochs=20)`

Prépare l’entraînement puis lance l’apprentissage.  
La perte `sparse_categorical_crossentropy` convient parce que les labels sont des IDs entiers.  
La métrique mesure à quelle fréquence le bon caractère est prédit.

Construisez un second modèle Keras destiné à la génération auto-régressive :
- entrée : un seul token (shape (1,)),
- entrée : l’état caché GRU,
- sortie : distribution softmax sur le vocabulaire + nouvel état.  
Copiez ensuite les poids du modèle entraîné vers ce modèle de génération.

In [ ]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

`generation_model = keras.Model(...)`
`generation_model.set_weights(model.get_weights())`

Construit un second modèle dédié à la génération caractère par caractère.  
Contrairement au modèle d’entraînement, il prend un seul token à la fois ainsi que l’état courant du GRU.  
On copie ensuite les poids du modèle entraîné pour réutiliser ce qu’il a appris.

```python
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
```

* `return_state=True` signifie que la couche renvoie son **état interne**
* `initial_state=` attend un tenseur ayant exactement la forme de l’état du GRU
* Le tenseur `state` que nous créons est passé comme `initial_state`
* Le `state` récupéré après `predict()` est précisément l’état interne mis à jour

Donc la correspondance est garantie par la construction même du modèle.

---

Un GRU a **un seul état interne**.

Si nous utilisions un **LSTM**, il faudrait initialiser **deux** états :

* l’état caché ( h )
* l’état cellule ( c )

Le fait qu’un seul tenseur soit initialisé confirme que nous utilisons bien un GRU.

Le GRU démarre ici à partir d’un état fourni explicitement.  
Ça permet de continuer une séquence déjà commencée sans tout recalculer depuis zéro à chaque pas.  
`output_state` est simplement le nouvel état mis à jour après lecture du token courant.

À partir du tokenizer, récupérez le vocabulaire et construisez :
- un dictionnaire `char_to_id`,
- un dictionnaire `id_to_char`.  
Définissez ensuite un prompt multi-lignes (ex : début de pièce “KING RICHARD III:”).

In [ ]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

`tokens = tokenizer.get_vocabulary()`
`char_to_id = ...`
`id_to_char = ...`

Construit les tables de correspondance entre caractères et IDs.  
Elles servent dans les deux sens :
- encoder un prompt en entiers,
- décoder les prédictions du modèle en texte lisible.

`prompt = """ ... """`

Définit le texte de départ qui servira d’amorce à la génération.  
Le modèle va s’appuyer dessus pour produire la suite.

Convertissez le prompt en IDs, initialisez un état GRU nul, puis faites passer les tokens du prompt un par un dans le modèle de génération afin d’obtenir un état interne cohérent avant de générer de nouveaux caractères.

In [ ]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

`input_ids = [char_to_id[c] for c in prompt]`
`state = keras.ops.zeros(...)`
`for token_id in input_ids: ...`

Transforme le prompt en IDs, initialise un état nul, puis fait passer les caractères du prompt un par un dans le modèle de génération.  
Le but est de “chauffer” l’état interne du GRU pour qu’il contienne déjà le contexte du prompt avant de générer de nouveaux caractères.

Lorsqu’on commence la génération, le modèle n’a encore rien lu.
Il faut donc lui donner un **état initial**.

Dans le code :

```python
state = keras.ops.zeros(shape=(1, hidden_dim))
```

* `1` → on génère une seule séquence (batch size = 1)
* `hidden_dim` → taille de l’état interne du GRU
* `zeros` → état initial neutre (aucune information mémorisée)

Le `1` correspond à la taille de batch, donc ici une seule séquence générée.

Générez une séquence de longueur fixée (ex : 250 caractères) en :
- choisissant à chaque pas le token le plus probable (argmax),
- réinjectant ce token dans le modèle,
- mettant à jour l’état.  
Decodez ensuite les IDs générés en caractères, et affichez `prompt + texte_généré`.

In [ ]:
import numpy as np

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

`for i in range(max_length): ...`
`next_char = int(np.argmax(...))`
`output = "".join(...)`

Boucle de génération auto-régressive.  
À chaque étape, on prend le caractère le plus probable avec `argmax`, on le réinjecte dans le modèle, on récupère le nouvel état, puis on recommence.  
À la fin, on reconvertit les IDs générés en caractères pour afficher le texte.

In [ ]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

## Traduction anglais → espagnol (seq2seq)

Téléchargez le dataset spa-eng (anglais–espagnol), extrayez l’archive, et construisez le chemin vers le fichier texte contenant les paires de phrases.

In [ ]:
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

Lisez le fichier ligne par ligne.  
Pour chaque ligne :
- séparez l’anglais et l’espagnol,
- ajoutez à l’espagnol un token `[start]` au début et `[end]` à la fin,  
puis stockez les paires dans une liste.

In [ ]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

`for line in lines:`
`english, spanish = line.split("\t")`
`spanish = "[start] " + spanish + " [end]"`

Lit chaque ligne du dataset et sépare la phrase anglaise de la phrase espagnole.  
On ajoute `[start]` et `[end]` à la phrase cible pour aider le décodeur à savoir quand commencer et quand s’arrêter.

Affichez une paire (anglais, espagnol) tirée aléatoirement afin de vérifier le parsing du dataset.

In [ ]:
import random
random.choice(text_pairs)

Mélangez les paires puis découpez-les en trois ensembles :
- entraînement,
- validation,
- test,  
avec environ 15% validation et 15% test.

In [ ]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

`random.shuffle(text_pairs)`
`train_pairs = ...`
`val_pairs = ...`
`test_pairs = ...`

Mélange les données puis les sépare en trois ensembles.  
On garde une partie pour apprendre, une partie pour contrôler pendant l’entraînement, et une partie finale pour tester.

Créez une fonction de standardisation qui met en minuscules et retire la ponctuation (en conservant les tokens spéciaux).  
Ensuite :
- créez un `TextVectorization` pour l’anglais (int, longueur fixe),
- un `TextVectorization` pour l’espagnol (int, longueur fixe + 1, avec standardisation personnalisée),  
et adaptez-les sur les textes d’entraînement.

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

`def custom_standardization(...)`
`english_tokenizer = ...`
`spanish_tokenizer = ...`
`adapt(...)`

Prépare deux tokenizers :
- un pour l’anglais,
- un pour l’espagnol.

La fonction de standardisation met en minuscules et enlève la ponctuation inutile tout en conservant les tokens spéciaux comme `[start]` et `[end]`.  
`adapt` apprend ensuite le vocabulaire à partir des textes d’entraînement.

Écrivez une fonction qui formate un batch (anglais, espagnol) en :
- `features`: un dict avec l’anglais tokenisé et l’espagnol tokenisé **sans le dernier token**,
- `labels`: l’espagnol tokenisé **sans le premier token** (décalage),
- `sample_weights`: masque qui ignore le padding (labels != 0).  
Puis créez une fonction qui construit un `tf.data.Dataset` (batch, map, shuffle, cache) à partir d’une liste de paires, et instanciez `train_ds` et `val_ds`.

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

`def format_dataset(eng, spa): ...`

Prépare les entrées et les cibles pour un modèle seq2seq.  
L’anglais devient l’entrée encodeur.  
L’espagnol est coupé en deux :
- entrée décodeur : tout sauf le dernier token,
- labels : tout sauf le premier token.

C’est exactement le principe du décalage enseignant la prédiction du mot suivant.  
`sample_weights` sert à ignorer le padding dans la perte et les métriques.

`def make_dataset(pairs): ...`
`train_ds = make_dataset(train_pairs)`
`val_ds = make_dataset(val_pairs)`

Construit les datasets TensorFlow à partir des paires de phrases.  
On batch, on transforme avec `format_dataset`, on mélange, puis on met en cache pour accélérer.

Récupérez un batch de `train_ds` et affichez les dimensions des tenseurs (anglais, espagnol, labels, sample_weights) pour vérifier la mise en forme.

In [ ]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

`inputs, targets, sample_weights = next(iter(train_ds))`

Récupère un batch pour vérifier les formes des tenseurs.  
C’est une étape utile pour s’assurer que l’encodeur, le décodeur, les labels et le masque padding sont bien alignés.

In [ ]:
print(inputs["spanish"].shape)

In [ ]:
print(targets.shape)

In [ ]:
print(sample_weights.shape)

## Seq2seq avec RNN (GRU)

Construisez l’encodeur du modèle seq2seq :
- entrée : séquence d’IDs anglais,
- `Embedding` avec `mask_zero=True`,
- `GRU` puis encapsulation en `Bidirectional` (fusion par somme),  
et récupérez la représentation finale de la phrase source.

In [ ]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

`source = keras.Input(...)`
`x = layers.Embedding(..., mask_zero=True)(source)`
`rnn_layer = layers.Bidirectional(layers.GRU(...), merge_mode="sum")`
`encoder_output = rnn_layer(x)`

Construit l’encodeur du modèle seq2seq.  
L’`Embedding` transforme les IDs anglais en vecteurs.  
`mask_zero=True` indique que le token `0` est du padding à ignorer.  
Le `Bidirectional` lit la phrase dans les deux sens, ce qui donne une représentation finale plus riche de la phrase source.

Construisez le décodeur :
- entrée : séquence d’IDs espagnols,
- `Embedding` avec masque,
- `GRU(return_sequences=True)` initialisée avec l’état de l’encodeur,
- `Dropout`,
- `Dense(vocab_size, softmax)` pour prédire le prochain token à chaque pas.  
Assemblez le modèle seq2seq complet.

In [ ]:
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

`target = keras.Input(...)`
`x = layers.Embedding(...)(target)`
`x = layers.GRU(..., return_sequences=True)(x, initial_state=encoder_output)`
`target_predictions = layers.Dense(..., activation="softmax")(x)`

Construit le décodeur.  
Il reçoit les tokens espagnols déjà connus et démarre à partir de l’état produit par l’encodeur.  
À chaque position, il prédit la distribution sur le prochain token espagnol.

`seq2seq_rnn = keras.Model([source, target], target_predictions)`

Assemble l’encodeur et le décodeur dans un seul modèle Keras prêt à être compilé et entraîné.

Affichez le résumé du modèle seq2seq (nombre de paramètres, formes de tenseurs).

In [ ]:
seq2seq_rnn.summary(line_length=80)

Compilez le modèle avec Adam et `sparse_categorical_crossentropy`, et entraînez-le sur `train_ds` en utilisant la validation `val_ds`.  
Assurez-vous que la métrique d’accuracy est calculée en tenant compte des poids d’échantillons (padding ignoré).

In [ ]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

`seq2seq_rnn.compile(...)`
`seq2seq_rnn.fit(train_ds, ..., validation_data=val_ds)`

Lance l’apprentissage de la traduction.  
`weighted_metrics=["accuracy"]` fait en sorte que l’accuracy tienne compte du masque de padding fourni dans le dataset.

Implémentez une fonction `generate_translation(sentence)` qui :
- tokenize la phrase anglaise,
- démarre une phrase cible avec `[start]`,
- prédit itérativement le prochain token (argmax) en réinjectant la cible partielle,
- s’arrête si `[end]` est généré ou si la longueur max est atteinte.  
Testez-la sur quelques phrases de l’ensemble de test.

## Transformer (encodeur/décodeur) + embeddings positionnels

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

`def generate_translation(input_sentence): ...`

Fait une traduction auto-régressive avec le modèle seq2seq.  
On encode la phrase anglaise, puis on commence la phrase cible avec `[start]`.  
Ensuite, à chaque étape, le modèle prédit le prochain token espagnol, qu’on réinjecte dans l’entrée du décodeur jusqu’à produire `[end]` ou atteindre la longueur max.

Créez une classe `TransformerEncoder(keras.Layer)` qui implémente :
- une self-attention multi-têtes,
- des connexions résiduelles,
- une normalisation de couche (LayerNorm),
- un MLP feed-forward (Dense ReLU puis Dense),
- une deuxième connexion résiduelle + LayerNorm,  
et qui prend en entrée un tenseur `source` et un masque `source_mask`.

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

`class TransformerEncoder(keras.Layer): ...`

Définit un bloc encodeur Transformer.  
Chaque token regarde tous les autres via la self-attention.  
Puis viennent :
- connexion résiduelle,
- normalisation,
- petit réseau feed-forward,
- deuxième résiduel + normalisation.

Le masque sert à empêcher l’attention de tenir compte du padding.

Créez une classe `TransformerDecoder(keras.Layer)` qui implémente :
- self-attention **causale** (masque causal),
- cross-attention sur la sortie de l’encodeur,
- connexions résiduelles,
- LayerNorm,
- feed-forward MLP,  
et qui prend en entrée `target`, `source`, `source_mask`.

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

`class TransformerDecoder(keras.Layer): ...`

Définit un bloc décodeur Transformer.  
Il fait d’abord une self-attention causale pour ne voir que les tokens déjà générés.  
Puis une cross-attention pour consulter la sortie de l’encodeur.  
Enfin, comme dans l’encodeur, il applique MLP, résiduels et normalisations.

Assemblez un modèle de traduction basé sur Transformer :
- Embedding sur l’anglais → `TransformerEncoder`,
- Embedding sur l’espagnol → `TransformerDecoder` (cross-attention sur l’encodeur),
- Dropout,
- Dense softmax de taille vocabulaire.  
Affichez le résumé, compilez (Adam + sparse_categorical_crossentropy + accuracy pondérée) et entraînez le modèle.

In [ ]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

`source = keras.Input(...)`
`x = layers.Embedding(...)(source)`
`encoder_output = TransformerEncoder(...)(...)`
`target = keras.Input(...)`
`x = layers.Embedding(...)(target)`
`x = TransformerDecoder(...)(...)`
`target_predictions = layers.Dense(..., activation="softmax")(x)`

Assemble un modèle de traduction basé sur Transformer.  
L’encodeur produit une représentation de la phrase anglaise, puis le décodeur génère la phrase espagnole en consultant cette représentation par cross-attention.

In [ ]:
transformer.summary(line_length=80)

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

`transformer.summary(...)`
`transformer.compile(...)`
`transformer.fit(...)`

Résumé, compilation et entraînement du modèle Transformer.  
Même logique que pour le seq2seq RNN, sauf que l’architecture repose ici sur l’attention plutôt que sur une lecture récurrente.

Créez une couche `PositionalEmbedding` qui :
- apprend un embedding des tokens,
- apprend un embedding des positions,
- additionne les deux pour produire la représentation finale.  
Les positions doivent être construites dynamiquement à partir de la longueur de la séquence en entrée.

In [ ]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

`class PositionalEmbedding(keras.Layer): ...`

Ajoute explicitement la notion de position aux embeddings.  
Un Transformer ne lit pas la séquence dans l’ordre comme un RNN, donc il faut lui donner une information de position.  
Cette couche apprend à la fois :
- un embedding du token,
- un embedding de sa position,
puis additionne les deux.

Reconstruisez le modèle Transformer de traduction en remplaçant les `Embedding` simples par votre `PositionalEmbedding` côté encodeur et décodeur.  
Compilez et entraînez le modèle sur davantage d’époques (ex : 30) afin d’améliorer les performances.

In [ ]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

`x = PositionalEmbedding(...)(source)`
`x = PositionalEmbedding(...)(target)`

Remplace les embeddings simples par des embeddings positionnels côté encodeur et décodeur.  
C’est important pour que le Transformer sache distinguer un mot au début d’un mot à la fin.

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

`transformer.fit(train_ds, epochs=30, validation_data=val_ds)`

Réentraîne le Transformer plus longtemps avec embeddings positionnels pour améliorer les performances.

### Comment un token calcule l’importance des autres tokens (attention)

Dans un Transformer, chaque token est d’abord transformé en **vecteur numérique** (embedding).  
À partir de ce vecteur, le modèle calcule trois nouveaux vecteurs :

- **Query (Q)** : ce que le token cherche
- **Key (K)** : ce que chaque token peut offrir comme information
- **Value (V)** : l’information réelle portée par le token

Ces vecteurs sont obtenus par de simples **transformations linéaires** (multiplications par des matrices apprises).

---

### Étape 1 — Comparer les tokens

Pour savoir quels tokens sont importants, le modèle compare la **query d’un token** avec les **keys de tous les tokens**.

Mathématiquement :

- \(i\) : token qui regarde
- \(j\) : token regardé
- produit scalaire = mesure de similarité

Si le score est grand → le token \(j\) est important pour \(i\).

---

### Étape 2 — Transformer les scores en poids

Les scores sont transformés en **probabilités** avec une fonction *softmax*.

Ces valeurs indiquent **combien d’attention le token \(i\) doit porter au token \(j\)**.

Exemple :

| token regardé | poids d’attention |
|---|---|
| I | 0.05 |
| love | 0.20 |
| deep | 0.60 |
| learning | 0.15 |

Ici, le token regarde surtout **deep**.

---

### Étape 3 — Combiner l’information

Le token construit ensuite une **nouvelle représentation** en faisant une moyenne pondérée des **values**.

Donc la représentation finale d’un token devient un **mélange d’informations provenant des autres tokens**, pondéré par leur importance.

---

Tous ces calculs (produits scalaires, softmax, combinaisons) peuvent être faits **avec des opérations matricielles**.

Cela signifie que :

- on compare **tous les tokens entre eux en même temps**
- aucun calcul ne dépend du résultat du token précédent

Donc la phrase n’a pas besoin d’être lue **mot par mot** comme dans un RNN.

Un Transformer fonctionne un peu comme si **chaque mot posait la question :**

> « Quels autres mots de la phrase sont les plus utiles pour comprendre mon rôle ? »

Puis il combine ces informations pour produire une représentation plus riche.

Adaptez la fonction de décodage auto-régressif `generate_translation` pour utiliser le modèle Transformer :
- tokenisez l’anglais,
- construisez itérativement la séquence cible,
- à chaque pas, prédisez le token suivant (argmax),
- stoppez sur `[end]`.  
Testez sur quelques phrases.

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

`generate_translation` version Transformer

Le principe reste le même que pour le modèle RNN :  
on part de `[start]`, on prédit le prochain token, on l’ajoute à la cible partielle, puis on recommence jusqu’à `[end]`.  
La différence, c’est qu’à chaque étape le Transformer recalcule la sortie à partir de toute la cible générée jusque-là, au lieu de propager un état caché comme un GRU.

## Classification avec Transformer pré-entraîné (RoBERTa) + IMDb

Chargez un tokenizer et l’architecture “backbone” d’un Transformer pré-entraîné (par exemple RoBERTa base en anglais) via `keras_hub` à partir d’un preset.

In [ ]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

`tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")`
`backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")`

Charge un tokenizer RoBERTa déjà prêt ainsi que le backbone pré-entraîné correspondant.  
Le tokenizer transforme le texte en sous-mots, et le backbone fournit les représentations contextualisées apprises pendant le pré-entraînement.

Testez le tokenizer sur une phrase simple (ex : “The quick brown fox”) puis affichez `summary()` du backbone pour inspecter l’architecture.

In [ ]:
tokenizer("The quick brown fox")

In [ ]:
backbone.summary(line_length=80)

Téléchargez le dataset IMDb (reviews), extrayez-le, puis créez des dossiers `train`, `val`, `test` en copiant les fichiers.  
Constituez la validation en prélevant une fraction (ex : 20%) des fichiers d’entraînement, séparément pour `pos` et `neg`, avec un shuffle reproductible.

In [ ]:
import os, pathlib, shutil, random

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

La validation est créée en prélevant une partie des exemples d’entraînement, séparément pour les avis positifs et négatifs, avec un mélange reproductible.

Créez `train_ds`, `val_ds`, `test_ds` à partir des répertoires en utilisant `text_dataset_from_directory`, avec une taille de batch (ex : 16).

In [ ]:
from keras.utils import text_dataset_from_directory

batch_size = 16
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

`train_ds = text_dataset_from_directory(...)`
`val_ds = ...`
`test_ds = ...`

Construit des datasets texte directement à partir de l’arborescence de dossiers.  
Les labels sont déduits automatiquement des sous-dossiers `pos` et `neg`.

Écrivez une fonction `preprocess(text, label)` qui :
- tokenize le texte avec le tokenizer RoBERTa,
- “pack” la séquence en ajoutant les tokens de début/fin, en paddant à une longueur fixe (ex : 512),
- renvoie un dictionnaire `{token_ids, padding_mask}` et le label.  
Appliquez-la à `train_ds`, `val_ds`, `test_ds` avec `.map()`.

In [ ]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

`def preprocess(text, label): ...`

Prétraite chaque review pour RoBERTa.  
Le tokenizer découpe le texte en tokens, puis `StartEndPacker` ajoute les tokens spéciaux de début/fin, tronque ou complète à une longueur fixe, et produit aussi un masque de padding.  
La sortie a le format attendu par le backbone.

`preprocessed_train_ds = train_ds.map(preprocess)`

Applique ce prétraitement à tous les datasets pour obtenir des entrées prêtes pour le modèle pré-entraîné.

Affichez un élément (ou batch) du dataset prétraité pour vérifier la présence des clés `token_ids` et `padding_mask` ainsi que le label associé.

In [ ]:
next(iter(preprocessed_train_ds))

`next(iter(preprocessed_train_ds))`

Permet de vérifier qu’un batch contient bien :
- `token_ids`,
- `padding_mask`,
- le label associé.

#### Fine-tuning a pretrained Transformer

Construisez un classifieur binaire en réutilisant le backbone :
- prenez les entrées du backbone,
- récupérez la représentation du token `[CLS]` (position 0),
- appliquez Dropout, Dense (ReLU), Dropout,
- sortie Dense(1, sigmoid).  
Créez le modèle Keras final.

In [ ]:
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

`inputs = backbone.input`
`x = backbone(inputs)`
`x = x[:, 0, :]`
`x = layers.Dropout(...)(x)`
`x = layers.Dense(768, activation="relu")(x)`
`outputs = layers.Dense(1, activation="sigmoid")(x)`

Construit un classifieur binaire au-dessus de RoBERTa.  
On récupère la représentation du premier token, souvent utilisée comme résumé global de la séquence.  
Puis on ajoute une petite tête de classification pour prédire positif ou négatif.

Compilez le modèle avec Adam (petit taux d’apprentissage, ex : 5e-5), `binary_crossentropy` et `accuracy`.  
Entraînez-le sur `preprocessed_train_ds` avec validation, puis évaluez-le sur `preprocessed_test_ds`.

In [ ]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
)

In [ ]:
classifier.evaluate(preprocessed_test_ds)

`classifier.compile(...)`
`classifier.fit(...)`
`classifier.evaluate(...)`

Compile le modèle, l’entraîne sur IMDb puis l’évalue sur le jeu de test.  
Le faible taux d’apprentissage est important en fine-tuning pour ne pas détruire trop brutalement les poids pré-entraînés.